# Chapter 7 - One-Hot Encoding & Word Embeddings

Machines can't read "dog", "cat", "love". We must convert discrete, categorical data into numbers. This chapter explains the two dominant strategies:

1. **One-hot encoding**: a vector with a single 1 among many 0s per category.
2. **Learned embeddings**: a small dense numeric vector per word *learned* by the network during training.

Both are fundamental; embeddings are what power modern NLP.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers, models

print("Keras", keras.__version__)

Keras 3.15.1


## Part 1 - One-hot by hand (the crisp, the clear, the naive)
For 4 categories (red, green, blue, yellow), index them and create a 4-dim one-hot vector.

In [2]:
categories = ["red", "green", "blue", "yellow"]
word_to_index = {w: i for i, w in enumerate(categories)}
samples = ["red green blue blue yellow", "blue green"]

def one_hot_text(text, vocab, vocab_size):
    vectors = np.zeros((len(text.split()), vocab_size))
    for i, w in enumerate(text.split()):
        vectors[i, vocab[w]] = 1.0
    return vectors

v = one_hot_text("red green blue blue", word_to_index, len(categories))
print("one-hot for 'red green blue blue':")
print(v)
print("column = category presence (each time the word is present it 1's its column)")

one-hot for 'red green blue blue':
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]
column = category presence (each time the word is present it 1's its column)


In [3]:
from keras.utils import to_categorical

labels = ["red", "blue", "green", "yellow", "yellow"]
label_ids = np.array([word_to_index[l] for l in labels])

onehot = keras.ops.one_hot(label_ids, len(categories))
print("keras.ops.one_hot(labels) shape:", onehot.shape)
print(onehot.numpy())

keras.ops.one_hot(labels) shape: (5, 4)
[[1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]]


In [4]:
onehot2 = to_categorical(label_ids, len(categories))
print("keras.utils.to_categorical(labels) shape:", onehot2.shape)
print(onehot2)

keras.utils.to_categorical(labels) shape: (5, 4)
[[1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]]


## Part 2 - Learned embeddings (the modern, dense way)
One-hot vectors are sparse (mostly zeros).
Embeddings replace them with a **learned dense vector** per word. In Keras this is the `Embedding(vocab_size, embedding_dim)` layer: index -> vector, and gradients flow back during training.

In [5]:
word_index = ["hello", "world", "deep", "learning", "is", "great"]
pad = 0
seq = np.array([[1, 2, 3, 4, 5, 6]])  # stopwords hidden for clarity

embed = layers.Embedding(input_dim=len(word_index)+1, output_dim=8, input_length=6)
inp = layers.Input(shape=(6,))
out = embed(inp)
em = models.Model(inp, out)
vector = em.predict(seq)
print("embedding output for the 6 words -> shape", vector.shape)
print(vector[0])
print("each row is a dense 8-dim vector; similar words will end up close in this space")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step


embedding output for the 6 words -> shape (1, 6, 8)
[[-0.01779892  0.03699077 -0.03526186 -0.04657587 -0.03416133  0.02057825
   0.02941957 -0.00197373]
 [-0.04269062 -0.02810543  0.00541357  0.03463401  0.0256835  -0.00401009
   0.0375734   0.03302914]
 [-0.03605419  0.01155193  0.01357255 -0.03705402  0.01380951  0.04516132
  -0.02203894  0.04888235]
 [-0.00798601  0.02584836  0.04418195  0.03177483  0.01423551 -0.02751675
  -0.02097145 -0.04137452]
 [ 0.01829721  0.02144656 -0.00432916  0.02498091 -0.04798905 -0.01725812
  -0.01696662 -0.01936426]
 [ 0.03711803  0.0077365   0.04626462  0.04477109  0.03318459  0.00206907
   0.01799697  0.04552061]]
each row is a dense 8-dim vector; similar words will end up close in this space


C:\Users\mrsur\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


## Embeddings as features: a mini sentiment classifier
Train an embedding-backed classifier on IMDB -- the baseline that the RNN chapter then beats with sequence modelling.

In [6]:
from keras.datasets import imdb
from keras.utils import pad_sequences

vocab_size = 10000
maxlen = 100
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)
print("shapes", x_train.shape, x_test.shape)

shapes (25000, 100) (25000, 100)


In [7]:
model = models.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(vocab_size, 8),
    layers.GlobalAveragePooling1D(),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 8)         │        80,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 8)              │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 80,009 (312.54 KB)

 Trainable params: 80,009 (312.54 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history = model.fit(x_train, y_train, validation_data=(x_test, y_test),
                    epochs=10, batch_size=256, verbose=1)
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"embedding baseline accuracy: {acc:.4f}")

Epoch 1/10


 1/98 ━━━━━━━━━━━━━━━━━━━━ 1:02 646ms/step - accuracy: 0.5039 - loss: 0.6928

25/98 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6047 - loss: 0.6907    

51/98 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5921 - loss: 0.6882

77/98 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6229 - loss: 0.6855

...

27/98 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8403 - loss: 0.4256 

53/98 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8411 - loss: 0.4249

78/98 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8407 - loss: 0.4230

98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8410 - loss: 0.4219 - val_accuracy: 0.8229 - val_loss: 0.4331


embedding baseline accuracy: 0.8229


## Recap
- **One-hot**: fast to build, sparse, huge, no semantic similarity.
- **Embeddings**: dense, trained for the task, capture meaning.
- Next step: stack an RNN on top of embeddings (chapter 8).